# Full Build Workflow Examples

This notebook shows two ways to use the legacy artifact-producing `BuildCompiler.full_build(...)` workflow:

1. A level-1 design that assembles successfully in one pass, then generates transformation and plating artifacts.
2. A level-2 design example that triggers the recovery stack: assembly level 2, assembly level 1, domestication, transformation, and plating.

`full_build(...)` returns a dictionary with `zip_path` / `artifact_zip`. The zip contains the manifest and all generated PUDU input JSON/scripts.

## Imports and paths

In [ ]:
from pathlib import Path
import json
import zipfile
from unittest.mock import patch

import sbol2

from buildcompiler.abstract_translator import extract_toplevel_definition
from buildcompiler.buildcompiler import BuildCompiler
from buildcompiler.constants import ENGINEERED_PLASMID

In [ ]:
ROOT = Path.cwd()
if not (ROOT / "tests" / "test_files").exists():
    ROOT = ROOT.parent

TEST_FILES = ROOT / "tests" / "test_files"
RESULTS_DIR = ROOT / "notebooks" / "results" / "full_build_workflow_examples"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

def load_sbol(filename):
    doc = sbol2.Document()
    doc.read(str(TEST_FILES / filename))
    return doc

def list_zip(zip_path):
    with zipfile.ZipFile(zip_path, "r") as archive:
        return sorted(archive.namelist())

def show_manifest(result):
    manifest = json.loads(Path(result["manifest_path"]).read_text())
    print(json.dumps({
        "zip_path": manifest["zip_path"],
        "domestication": manifest["domestication"],
        "assembly_lvl1": manifest["assembly_lvl1"],
        "assembly_lvl2": manifest["assembly_lvl2"],
        "transformation_count": len(manifest["transformation"]["successful"]),
        "plating_count": len(manifest["plating"]["successful"]),
    }, indent=2))

print(TEST_FILES)
print(RESULTS_DIR)

## Load local SBOL inventory fixtures

The first example uses the local SBOL fixtures in `tests/test_files` so it can run without SynBioHub credentials.

In [ ]:
collection_docs = [
    load_sbol("CIDARMoCloParts_collection.xml"),
    load_sbol("CIDARMoCloPlasmidsKit_collection.xml"),
    load_sbol("Enzyme_Implementations_collection.xml"),
    load_sbol("impl_test_collection.xml"),
]

len(collection_docs)

## Example 1: level-1 design succeeds in one pass

This runs a direct level-1 full build. Since the fixture inventory contains the required part plasmids and backbone, the build should go straight through assembly level 1, transformation, plating, manifest writing, and zip packaging. Level 2 is recorded as skipped because this input is a level-1 design.

In [ ]:
lvl1_design_doc = load_sbol("abstract_design.xml")
lvl1_design = extract_toplevel_definition(lvl1_design_doc)

lvl1_compiler = BuildCompiler.from_local_documents(
    collection_docs,
    design_doc=lvl1_design_doc,
)

lvl1_result = lvl1_compiler.full_build(
    designs=[lvl1_design],
    results_dir=RESULTS_DIR / "lvl1_success",
    overwrite=True,
)

print(lvl1_result["zip_path"])
show_manifest(lvl1_result)

In [ ]:
list_zip(lvl1_result["zip_path"])

## Example 2: level-2 design triggers all stages

The recovery path depends on the available inventory. To make the stage sequence deterministic in a notebook, this example uses a tiny SBOL level-2 design and patches the stage methods to simulate the exact condition we want to demonstrate:

1. The first level-2 assembly attempt fails because level-1 regions are missing.
2. The first level-1 assembly attempt fails because a part needs domestication.
3. Domestication produces a part plasmid and PUDU assembly payload.
4. Level-1 assembly retries successfully.
5. Level-2 assembly retries successfully.
6. Transformation and plating artifacts are produced and packaged.

For a real run, remove the patches and provide SBOL collections/designs that naturally produce this dependency stack.

In [ ]:
def make_plasmid(doc, display_id):
    plasmid = sbol2.ComponentDefinition(display_id)
    plasmid.roles = [ENGINEERED_PLASMID]
    doc.add(plasmid)
    return plasmid

def make_lvl2_doc():
    doc = sbol2.Document()
    tu = sbol2.ComponentDefinition("demo_tu")
    lvl2 = sbol2.ComponentDefinition("demo_lvl2_design")
    doc.add(tu)
    doc.add(lvl2)
    comp = lvl2.components.create("demo_tu_component")
    comp.definition = tu.identity
    return doc

trigger_doc = sbol2.Document()
trigger_compiler = BuildCompiler.from_local_documents([], design_doc=trigger_doc)

missing_part = make_plasmid(trigger_compiler.sbol_doc, "missing_promoter")
domesticated = make_plasmid(trigger_compiler.sbol_doc, "domesticated_missing_promoter")
lvl1_product = make_plasmid(trigger_compiler.sbol_doc, "assembled_demo_tu")
lvl2_product = make_plasmid(trigger_compiler.sbol_doc, "assembled_demo_lvl2")
lvl2_design_doc = make_lvl2_doc()

stage_calls = []

In [ ]:
def fake_assembly_lvl2(*args, **kwargs):
    stage_calls.append("assembly_lvl2")
    if stage_calls.count("assembly_lvl2") == 1:
        raise RuntimeError("level-2 input is missing level-1 regions")
    trigger_compiler.last_assembly_pudu_json_by_stage = {
        "assembly_lvl2": [
            {
                "Product": lvl2_product.identity,
                "Backbone": "demo_lvl2_backbone",
                "PartsList": [lvl1_product.identity],
                "Restriction Enzyme": "BbsI",
            }
        ]
    }
    return [lvl2_product], trigger_compiler.sbol_doc

def fake_assembly_lvl1(*args, **kwargs):
    stage_calls.append("assembly_lvl1")
    if stage_calls.count("assembly_lvl1") == 1:
        raise RuntimeError("level-1 input is missing a domesticated part")
    trigger_compiler.last_assembly_pudu_json_by_stage = {
        "assembly_lvl1": [
            {
                "Product": lvl1_product.identity,
                "Backbone": "demo_lvl1_backbone",
                "PartsList": [domesticated.identity],
                "Restriction Enzyme": "BsaI",
            }
        ]
    }
    return [lvl1_product], trigger_compiler.sbol_doc

def fake_domestication(parts):
    stage_calls.append("domestication")
    trigger_compiler.last_assembly_pudu_json_by_stage = {
        "domestication": [
            {
                "Product": domesticated.identity,
                "Backbone": "demo_domestication_backbone",
                "PartsList": [missing_part.identity],
                "Restriction Enzyme": "BsaI",
            }
        ]
    }
    return [domesticated]

def fake_transformation(products, chassis_name="E_coli_DH5alpha", transformation_doc=None):
    stage_calls.append("transformation")
    product_id = products[0].identity
    return {
        "stage": "transformation",
        "chassis": chassis_name,
        "sbol_artifacts": [
            {
                "transformed_strain_module": f"{product_id}_strain",
                "transformed_strain_implementation": f"{product_id}_strain_impl",
            }
        ],
    }

def fake_plating(*args, **kwargs):
    stage_calls.append("plating")
    return {
        "stage": "plating",
        "json_intermediate": {
            "plating_data": {
                "bacterium_locations": {"A1": "demo_transformed_strain"}
            }
        },
    }

In [ ]:
with patch.object(trigger_compiler, "assembly_lvl2", side_effect=fake_assembly_lvl2), \
     patch.object(trigger_compiler, "assembly_lvl1", side_effect=fake_assembly_lvl1), \
     patch.object(trigger_compiler, "domestication", side_effect=fake_domestication), \
     patch.object(trigger_compiler, "transformation", side_effect=fake_transformation), \
     patch.object(trigger_compiler, "plating", side_effect=fake_plating), \
     patch.object(trigger_compiler, "_find_missing_parts_for_lvl1", return_value=[{"part": missing_part}]):
    lvl2_result = trigger_compiler.full_build(
        designs=lvl2_design_doc,
        results_dir=RESULTS_DIR / "lvl2_triggers_all_stages",
        overwrite=True,
    )

print(stage_calls)
print(lvl2_result["zip_path"])
show_manifest(lvl2_result)

In [ ]:
list_zip(lvl2_result["zip_path"])

## Expected PUDU artifacts

For the level-2 staged example, the zip should include at least these PUDU-facing files:

- `assembly_lvl2_pudu_assembly_input.json`
- `assembly_lvl1_pudu_assembly_input.json`
- `domestication_pudu_assembly_input.json`
- `pudu_transformation_protocol.py`
- `transformation_pudu_input.json`
- `transformation_plasmid_locations.json`
- `pudu_plating_protocol.py`
- `plating_pudu_input.json`
- `full_build_manifest.json`

In [ ]:
expected = {
    "assembly_lvl2_pudu_assembly_input.json",
    "assembly_lvl1_pudu_assembly_input.json",
    "domestication_pudu_assembly_input.json",
    "pudu_transformation_protocol.py",
    "transformation_pudu_input.json",
    "transformation_plasmid_locations.json",
    "pudu_plating_protocol.py",
    "plating_pudu_input.json",
    "full_build_manifest.json",
}

names = set(list_zip(lvl2_result["zip_path"]))
missing = sorted(expected - names)
print("Missing expected artifacts:", missing)
assert not missing